# Feature Store + ML Inference Service

This notebook demonstrates Snowflake's **Online Feature Store** (Postgres-backed) integrated with a real-time **ML Inference Service** using `feature_sources_per_function`.

### Why This Matters

In production ML, one of the hardest problems is **training-serving skew** — when features at inference time differ from those used during training due to separate code paths and data sources. The Feature Store + Inference Service integration solves this:

- **Single source of truth** — Same Feature View serves training and inference. No duplicate logic.
- **Decoupled clients** — API consumers send only an entity key. No need to know which features exist or how they're computed.
- **Low-latency** — Postgres online store provides millisecond-level feature retrieval for real-time workloads.
- **Auto-fresh** — Features sync from source to online store with configurable lag (10s in this demo).

### What We Build

1. A **Postgres-backed Feature Store** with online serving enabled
2. An **XGBoost churn model** trained on synthetic data, registered in the Model Registry
3. A **real-time inference service** (SPCS) that automatically fetches features from the online store when callers send only a `CUSTOMER_ID`

> **Note:** This demo uses synthetic customer churn data. The same pattern applies to fraud detection, recommendations, dynamic pricing, or any real-time ML use case.

In [ ]:
pip install snowflake-ml-python==1.44.0

## Part 1: Feature Store Setup

### Step 1: Create Database & Schema

Creates the `ML_DEMO_INF_FS_LOOKUP.ML_SERVICE_FS` database and schema to house the feature store, model, and inference service.

In [ ]:
%%sql -r setup_result
USE ROLE ACCOUNTADMIN;
CREATE DATABASE IF NOT EXISTS ML_DEMO_INF_FS_LOOKUP;
CREATE SCHEMA IF NOT EXISTS ML_DEMO_INF_FS_LOOKUP.ML_SERVICE_FS;

In [ ]:
%%sql -r use_schema
USE DATABASE ML_DEMO_INF_FS_LOOKUP;
USE SCHEMA ML_SERVICE_FS;

### Step 2: Generate Synthetic Data

Creates 200 synthetic customer records with features (`TENURE_MONTHS`, `MONTHLY_CHARGES`, `TOTAL_CHARGES`, `NUM_SUPPORT_TICKETS`, `CONTRACT_TYPE`) and a binary `CHURNED` target.

In [ ]:
# Generate a small synthetic customer churn dataset
import pandas as pd
import numpy as np

np.random.seed(42)
n_customers = 200

data = pd.DataFrame({
    "CUSTOMER_ID": [f"CUST_{i:04d}" for i in range(n_customers)],
    "TENURE_MONTHS": np.random.randint(1, 72, n_customers),
    "MONTHLY_CHARGES": np.round(np.random.uniform(20, 120, n_customers), 2),
    "TOTAL_CHARGES": np.round(np.random.uniform(100, 8000, n_customers), 2),
    "NUM_SUPPORT_TICKETS": np.random.randint(0, 10, n_customers),
    "CONTRACT_TYPE": np.random.choice([0, 1, 2], n_customers),  # 0=month-to-month, 1=one-year, 2=two-year
    "CHURNED": np.random.choice([0, 1], n_customers, p=[0.73, 0.27]),
})

print(f"Dataset shape: {data.shape}")
data.head()

### Step 3: Load Data into Snowflake

Gets the active Snowpark session, writes the synthetic DataFrame to a Snowflake table (`CUSTOMER_FEATURES`), which will serve as the source for the Feature View.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity

session = get_active_session()
session.use_database("ML_DEMO_INF_FS_LOOKUP")
session.use_schema("ML_SERVICE_FS")

# Write synthetic data to Snowflake
session.create_dataframe(data).write.save_as_table("CUSTOMER_FEATURES", mode="overwrite")
print("Table CUSTOMER_FEATURES created with", session.table("CUSTOMER_FEATURES").count(), "rows")

### Step 4: Initialize Feature Store & Register Entity

Initializes the Feature Store client, then registers a `CUSTOMER` entity keyed on `CUSTOMER_ID`. The entity defines how features are joined to inference requests.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode
from snowflake.ml.feature_store import OnlineConfig, OnlineStoreType

# Initialize the Feature Store
fs = FeatureStore(
    session=session,
    database="ML_DEMO_INF_FS_LOOKUP",
    name="ML_SERVICE_FS",
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

# Define entity
customer_entity = Entity(name="CUSTOMER", join_keys=["CUSTOMER_ID"])
fs.register_entity(customer_entity)
print("Entity registered:", customer_entity.name)

### Step 4b: Register Feature View with Postgres Online Store

Creates the `CUSTOMER_CHURN_FEATURES` Feature View with `OnlineStoreType.POSTGRES` enabled. This provisions a Postgres instance for low-latency key-value lookups with a 10-second target lag from the source table.

In [ ]:
# Create a Feature View with Postgres-backed online store
source_df = session.table("ML_DEMO_INF_FS_LOOKUP.ML_SERVICE_FS.CUSTOMER_FEATURES").select(
    "CUSTOMER_ID", "TENURE_MONTHS", "MONTHLY_CHARGES", 
    "TOTAL_CHARGES", "NUM_SUPPORT_TICKETS", "CONTRACT_TYPE"
)

cust_fv = FeatureView(
    name="CUSTOMER_CHURN_FEATURES",
    entities=[customer_entity],
    feature_df=source_df,
    desc="Customer churn prediction features",
    refresh_freq="1 minute",
    online_config=OnlineConfig(
        enable=True,
        target_lag="10s",
        store_type=OnlineStoreType.POSTGRES,
    ),
)

cust_fv = fs.register_feature_view(feature_view=cust_fv, version="V2", overwrite=True)
print(f"Feature View registered: {cust_fv.name} version {cust_fv.version}")
print(f"Online store type: POSTGRES")

### Step 5: Configure PAT & Wait for Online Sync

The Postgres online store requires a **Programmatic Access Token (PAT)** for authentication. This cell validates the PAT is available in the environment, then waits for the online store to sync data from the source table.

In [ ]:
import os, time

# PAT must be set as an environment variable on the notebook service.
# To configure:
#   1. Go to notebook Settings > Environment > Secrets
#   2. Add secret: key=SNOWFLAKE_PAT, reference=ML_DEMO_INF_FS_LOOKUP.ML_SERVICE_FS.ML_FS_PAT_SECRET
# Or generate a new PAT via SQL:
#   ALTER USER <user> ADD PROGRAMMATIC ACCESS TOKEN <name> COMMENT='...';
#   Then store as a Snowflake secret and bind to the service.

if 'SNOWFLAKE_PAT' not in os.environ:
    raise EnvironmentError(
        "SNOWFLAKE_PAT not found. Add it via notebook Settings > Secrets "
        "(reference the ML_DEMO_INF_FS_LOOKUP.ML_SERVICE_FS.ML_FS_PAT_SECRET secret)."
    )

print(f"PAT available (length: {len(os.environ['SNOWFLAKE_PAT'])} chars)")
print("Waiting for online store to sync...")
time.sleep(15)
print("Done.")

### Step 6: Test Online Feature Retrieval

Reads features directly from the Postgres online store using entity keys. This confirms the online store is synced and serving data at low latency.

In [ ]:
from snowflake.ml.feature_store.feature_view import StoreType

# Retrieve features from the POSTGRES online store
cust_fv = fs.get_feature_view("CUSTOMER_CHURN_FEATURES", "V2")
print(f"Store type: {cust_fv.online_config.store_type}")
print(f"Target lag: {cust_fv.online_config.target_lag}")

test_keys = [["CUST_0001"], ["CUST_0010"], ["CUST_0050"]]

online_result = fs.read_feature_view(
    cust_fv,
    keys=test_keys,
    store_type=StoreType.ONLINE
)
print("\nOnline (Postgres) feature retrieval successful!")
online_result

### Step 7: Verify Feature Store State

Lists all registered entities and feature views in the feature store to confirm the setup is complete.

In [ ]:
# Verify: list all entities and feature views in this feature store
print("=== Registered Entities ===")
print(fs.list_entities().to_pandas())
print("\n=== Registered Feature Views ===")
fvs = fs.list_feature_views().to_pandas()
print(fvs[["NAME", "VERSION", "DESC"]].to_string())

---
## Part 2: Train, Register & Deploy

### Step 8: Train XGBoost Model

Trains a simple XGBoost binary classifier on the synthetic churn data. Uses an 80/20 train/test split and reports accuracy and ROC AUC.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# Features and target
FEATURES = ["TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES", "NUM_SUPPORT_TICKETS", "CONTRACT_TYPE"]
TARGET = "CHURNED"

X = data[FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost
model = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=4,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"ROC AUC:  {roc_auc_score(y_test, y_proba):.3f}")

### Step 9: Register Model in Snowflake

Logs the trained XGBoost model to the Snowflake Model Registry with its input signature (`sample_input_data`). This lets the inference service know which features the model expects.

In [ ]:
from snowflake.ml.registry import Registry
import pandas as pd

reg = Registry(session=session, database_name="ML_DEMO_INF_FS_LOOKUP", schema_name="ML_SERVICE_FS")

# Get the already-registered model version (or log a new one)
try:
    mv = reg.get_model("CHURN_XGBOOST").version("V1")
    print("Using existing model: CHURN_XGBOOST/V1")
except Exception:
    sample_input = X_train.head(5)
    mv = reg.log_model(
        model_name="CHURN_XGBOOST",
        version_name="V1",
        model=model,
        sample_input_data=sample_input,
        conda_dependencies=["xgboost"],
    )
    print("Model registered: CHURN_XGBOOST version V1")

funcs = mv.show_functions()
print(f"Functions: {[f['name'] if isinstance(f, dict) else str(f) for f in funcs]}")

### Step 10: Deploy Inference Service with Feature Store Lookup

Deploys the model as a real-time HTTP service on SPCS. The key parameter is `feature_sources_per_function` — it maps the `predict` method to the Postgres-backed Feature View so the service **automatically fetches features** when callers send only `CUSTOMER_ID`.

In [ ]:
# Get the registered feature view for online lookup
cust_fv = fs.get_feature_view("CUSTOMER_CHURN_FEATURES", "V2")

# Deploy with feature store integration
# feature_sources_per_function maps the "predict" function to the feature view
# so callers only need to pass CUSTOMER_ID and features are auto-fetched from the online Postgres store
mv.create_service(
    service_name="CHURN_PREDICTION_SVC",
    service_compute_pool="ML_ONLINE_CPU_POOL",
    ingress_enabled=True,
    feature_sources_per_function={"predict": [cust_fv]},
)
print("Service deployment initiated: CHURN_PREDICTION_SVC")
print("The service will auto-lookup features from CUSTOMER_CHURN_FEATURES/V2 (Postgres online store)")

In [ ]:
# Verify the service is running
services_df = mv.list_services()
print("Active services:")
print(services_df.to_string())

---
## Invoking the Inference Service

`feature_sources_per_function` enables the inference service to **automatically look up features** from the online Postgres Feature Store. Callers only need to send the entity key (`CUSTOMER_ID`) — the Snowflake ingress gateway enriches the request with all required features before forwarding to the model.

This enrichment happens at the **REST API layer** (public endpoint). To invoke the service, make an HTTP POST from any external client:

```bash
curl -X POST "https://<ENDPOINT_URL>/predict" \
  -H 'Authorization: Snowflake Token="<PAT_TOKEN>"' \
  -H 'Content-Type: application/json' \
  -d '{"dataframe_split": {"index": [0, 1, 2], "columns": ["CUSTOMER_ID"], "data": [["CUST_0001"], ["CUST_0010"], ["CUST_0050"]]}}'
```

**What happens under the hood:**
1. You send only `CUSTOMER_ID` values
2. The ingress gateway intercepts the request
3. It fetches `TENURE_MONTHS`, `MONTHLY_CHARGES`, `TOTAL_CHARGES`, `NUM_SUPPORT_TICKETS`, `CONTRACT_TYPE` from the Postgres online Feature Store
4. The enriched payload is forwarded to the XGBoost model container
5. Churn predictions are returned

**Note:** The Python SDK (`mv.run()`) and SQL service functions do not trigger this enrichment — they bypass the gateway. This feature is designed for external real-time inference clients (web apps, microservices, mobile backends) that call the REST endpoint.